## Workspace setup

In [ ]:
from datetime import datetime  
import uproot
from functools import partial
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

import importlib

# Notebooks run from VSCode use home directory as a base path
# while notebooks run from JupyterLab use the current directory as a base path
import sys
sys.path.append("/home/akalinow/scratch/ELITPC/TPCReco/PythonAnalysis/python/")

# Change directory to the working directory
import os
os.chdir('/scratch_hdd/akalinow/ELITPC/PythonAnalysis/')

## Training dataset preparation

In [ ]:
import io_functions as io
importlib.reload(io)

batchSize = 32
dataset = tf.data.Dataset.load('MergedEvent_Track3D_TwoProng_gun_MC_200k_filtered_length_30-100mm', compression="GZIP")
dataset = dataset.batch(batchSize, drop_remainder=True)
dataset = dataset.map(lambda x: x['sim'])
dataset = dataset.map(lambda x,y: (tf.reshape(x, (-1,)+io.projections.shape), tf.reshape(y, (-1,9))))
dataset = dataset.prefetch(tf.data.AUTOTUNE)
#tfds.benchmark(dataset.take(1000),batch_size=batchSize)

## Model definition

In [ ]:
def getModel():

  model = tf.keras.Sequential([
  tf.keras.layers.Input(shape=(256,512,3), name="input_image", dtype=tf.float32),
  tf.keras.layers.Resizing(height=256, width=256), 
  tf.keras.layers.Conv2D(16, 4, padding='same', activation='relu', 
                          data_format="channels_last"),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Conv2D(32, 2, padding='same', activation='relu', 
                          data_format="channels_last"),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Conv2D(64, 2, padding='same', activation='relu', 
                          data_format="channels_last"),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(9, activation="linear")
  ])

  initial_learning_rate = 0.01
  lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate,
                  decay_steps=836,
                  decay_rate=0.98,
                  staircase=False)

  optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule) 
  model.compile(optimizer = optimizer, 
                loss = 'mse', 
                metrics=['mse', 'mape']) 

  model.summary()
  return model
#########################################################
#########################################################  
def getModel_1():

  model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(256,512,3), name="input_image", dtype=tf.float32),
    tf.keras.layers.Resizing(height=256, width=256), 
    tf.keras.layers.Conv2D(16, kernel_size=(1,2), strides=(1,2), padding='valid', activation='relu', 
                          data_format="channels_last"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.LayerNormalization(),
    tf.keras.layers.Conv2D(32, 4, padding='same', activation='relu', 
                          data_format="channels_last"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.LayerNormalization(),
    tf.keras.layers.Conv2D(128, 2, padding='same', activation='relu', 
                          data_format="channels_last"),
    tf.keras.layers.LayerNormalization(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
    tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
    tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
    tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
    tf.keras.layers.Dense(9, activation="linear"),
  ])

  initial_learning_rate = 0.01
  lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate,
                  decay_steps=836,
                  decay_rate=0.98,
                  staircase=False)

  optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule) 
  model.compile(optimizer = optimizer, 
                loss = 'mse', 
                metrics=['mse', 'mape']) 

  model.summary()
  return model
##########################################################
##########################################################

In [22]:
model = getModel_1()
#model = getModelWithTL()

#model_path = "./training/0100_2025_Oct_13_18_48_39.keras"
#model = tf.keras.models.load_model(model_path)
#model.evaluate(dataset.take(10))
#model.summary()

#item = next(iter(dataset.take(1)))
#model(item)[0]

## Model training

In [24]:
%%time

import plotting_functions as plf
importlib.reload(plf)

log_dir = "logs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
#tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, profile_batch=(10, 20))
early_stop_callback = tf.keras.callbacks.EarlyStopping(patience=5, verbose=1)
callbacks =  [early_stop_callback] 

epochs=5
model = getModel_1()
model.trainable = True

initial_learning_rate = 0.001
decay_steps = 2*3891
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate,
                  decay_steps=decay_steps,
                  decay_rate=0.98,
                  staircase=False)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule) 
model.compile(optimizer = optimizer, 
                loss = 'mse', 
                metrics=['mse']) 

history = model.fit(dataset.skip(10), 
                        epochs=epochs,
                        verbose = 1,
                        validation_data = dataset.take(10),
                        #callbacks=callbacks
                        )
plf.plotTrainHistory(history)

current_time = datetime.now().strftime("%Y_%b_%d_%H_%M_%S")
print("Training start. Current Time =", current_time)

job_dir = f"training/{epochs:04d}_"+current_time+".keras"
model.save(job_dir)

job_dir = f"training/{epochs:04d}_"+current_time+"/"
model.export(job_dir)

## Model performance on training data.

Fill Pandas DataFrame with true and response values.

In [ ]:
%%time
import utility_functions as utils
importlib.reload(utils)
import pandas as pd
#model_path = "./training/0050_2025_Oct_14_15_14_48.keras"
#model_path = "./training/0020_2025_Nov_27_13_05_07.keras"
#model = tf.keras.models.load_model(model_path)

nBatches = 10_000
data = np.zeros_like(utils.getSimRecoColumns(utils.columnsXYZ).reshape(1,-1)) 

for aBatch in dataset.take(nBatches):

    features = aBatch[0]
    labels = aBatch[1].numpy()
    modelAnswer = model(features).numpy()
    data = np.append(data, np.column_stack((labels,modelAnswer)), axis=0)

df_XYZ = pd.DataFrame(data=data[1:], columns = utils.getSimRecoColumns(utils.columnsXYZ))
df_XYZ.describe()  

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

import utility_functions as utils
importlib.reload(utils)

for aBatch in dataset.skip(100).take(3):
    plf.plotEvent(aBatch, model=model)

### Resolution plots

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

#plf.controlPlots(df)
plf.plotEndPointRes(df=df_XYZ, edge="Vtx",coordinates=["x", "y", "z"])
plf.plotEndPointRes(df=df_XYZ, edge="Alpha",coordinates=["x", "y", "z"])
plf.plotEndPointRes(df=df_XYZ, edge="Carbon",coordinates=["x", "y", "z"])

plf.plotLengthPull(df_XYZ, partName="Alpha")
plf.plotLengthPull(df_XYZ, partName="Carbon")
plf.plotLengthPullEvolution(df_XYZ)
plf.plotOpeningAngleCos(df_XYZ)

## Various tests